# HAIO 2024 - Tű a Szénakazalban Megoldás

Ez a notebook a Magyar Mesterséges Intelligencia Olimpia (HAIO) 2024 "Tű a Szénakazalban" feladatának megoldását tartalmazza.

A feladat két részből áll:
1. **PyTorch rész**: RNN/CNN alapú bináris klasszifikáció bináris sorozatokon
2. **Mistral API rész**: LLM "Needle in a Haystack" tesztelés

A feladat Greg Kamradt "Needle in a Haystack" tesztjéből merít inspirációt.

## Telepítés és importok

In [ ]:
!pip install mistralai --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
import copy
from collections import defaultdict

# Reprodukálhatóság
torch.manual_seed(42)
np.random.seed(42)

# GPU ha elérhető
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Eszköz: {device}")

## Adatgenerálás

A `create_dataset` függvény bináris sorozatokat generál. A pozitív mintákba egy "tű" mintázat (`101010...`) van beillesztve véletlenszerű pozícióba, a negatív minták véletlenszerű bináris sorozatok.

In [ ]:
def create_dataset(n_pos, n_neg, seq_len, needle_len=20):
    """Bináris sorozatokból álló TensorDataset létrehozása.
    A pozitív minták tartalmazzák a tűt ('101010...') véletlenszerű pozícióban.
    A negatív minták véletlenszerű bináris sorozatok."""
    needle = torch.tensor([int(i % 2 == 0) for i in range(needle_len)], dtype=torch.float32)
    data = []
    labels = []
    for _ in range(n_pos):
        seq = torch.randint(0, 2, (seq_len,), dtype=torch.float32)
        pos = torch.randint(0, seq_len - needle_len + 1, (1,)).item()
        seq[pos:pos+needle_len] = needle
        data.append(seq)
        labels.append(1.0)
    for _ in range(n_neg):
        seq = torch.randint(0, 2, (seq_len,), dtype=torch.float32)
        data.append(seq)
        labels.append(0.0)
    data = torch.stack(data).unsqueeze(-1)  # (N, seq_len, 1)
    labels = torch.tensor(labels).unsqueeze(-1)  # (N, 1)
    return TensorDataset(data, labels)

In [ ]:
# Alap adathalmazok létrehozása
SEQ_LEN = 40
NEEDLE_LEN = 20

train_dataset = create_dataset(500, 500, SEQ_LEN, NEEDLE_LEN)
val_dataset = create_dataset(100, 100, SEQ_LEN, NEEDLE_LEN)
test_dataset = create_dataset(200, 200, SEQ_LEN, NEEDLE_LEN)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Tanító minták: {len(train_dataset)}, Validációs: {len(val_dataset)}, Teszt: {len(test_dataset)}")

## Az alap (hibás) RNN modell

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)
        out = self.fc(h_n[-1])
        return out

---
## 1. feladat: Költségfüggvény (5 pont)

Számítsuk ki a tanítatlan modell BCE veszteségét egy batch-en! Használjuk a `BCEWithLogitsLoss` függvényt.

In [ ]:
# Tanítatlan modell létrehozása
model_untrained = SimpleRNN().to(device)

# Költségfüggvény: BCEWithLogitsLoss (logitokra működik, nem kell sigmoid a modell végére)
criterion = nn.BCEWithLogitsLoss()

# Egy batch kiértékelése
batch_data, batch_labels = next(iter(train_loader))
batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)

with torch.no_grad():
    logits = model_untrained(batch_data)
    loss = criterion(logits, batch_labels)

print(f"Tanítatlan modell vesztesége: {loss.item():.4f}")
print(f"Várt érték (véletlenszerű tippelés): {-np.log(0.5):.4f} (= ln(2))")
print(f"\nA tanítatlan modell vesztesége közel van ln(2) ≈ 0.693-hoz,")
print(f"ami azt jelzi, hogy a modell véletlenszerűen tippel.")

---
## 2. feladat: False negative valószínűség (10 pont)

Mekkora annak a valószínűsége, hogy egy véletlenszerű bináris sorozat (hossza `l`) tartalmazza a `k` hosszúságú tű mintázatot?

### Képlet

A tű mintázat (`101010...`) egy meghatározott `k` bites sorozat. Egy adott pozíción a tű megjelenésének valószínűsége $(1/2)^k$.

Egy `l` hosszú sorozatban `l - k + 1` lehetséges kezdőpozíció van. Annak a valószínűsége, hogy a tű **nem** jelenik meg egyetlen adott pozíción sem:

$$P(\text{nincs tű egy pozíción}) = 1 - (1/2)^k$$

Ha a pozíciókat közelítőleg függetlennek tekintjük (ami jó közelítés nagy `l`-re és kis `k/l` arányra):

$$P(\text{nincs tű sehol}) \approx \left(1 - (1/2)^k\right)^{l-k+1}$$

$$P(\text{van tű}) \approx 1 - \left(1 - 2^{-k}\right)^{l-k+1}$$

**Megjegyzés**: Ez közelítés, mert a szomszédos pozíciók nem teljesen függetlenek. De nagy `k`-ra nagyon pontos.

Ez egyben a false negative ráta is: annak a valószínűsége, hogy egy negatívnak címkézett minta valójában tartalmazza a tűt.

In [ ]:
def false_negative_probability(seq_len, needle_len=20):
    """Annak a valószínűsége, hogy egy véletlenszerű bináris sorozat tartalmazza a tűt.
    Ez egyben a false negative ráta is a negatív mintákban."""
    n_positions = seq_len - needle_len + 1
    p_match_at_pos = 2 ** (-needle_len)  # Egy pozíción a tű megjelenésének valószínűsége
    p_no_match_anywhere = (1 - p_match_at_pos) ** n_positions
    p_contains_needle = 1 - p_no_match_anywhere
    return p_contains_needle


# Különböző sorozathosszakra
print("Sorozathossz | P(tartalmazza a tűt) | False negative ráta")
print("-" * 60)
for sl in [40, 80, 160, 320, 640, 1000, 10000, 100000, 1000000]:
    p = false_negative_probability(sl, NEEDLE_LEN)
    print(f"{sl:>12} | {p:>20.10e} | {p*100:.8f}%")

print(f"\nk=20-ra a tű megjelenési valószínűsége egy pozíción: {2**(-20):.2e}")
print(f"Tehát 40 hosszú sorozatban (21 pozíció): {false_negative_probability(40, 20):.2e}")
print(f"Ez rendkívül kicsi, a negatív mintáink gyakorlatilag biztosan nem tartalmazzák a tűt.")

---
## 3. feladat: Javíts a tanítókódon (5 pont)

A probléma az alap tanítással:
- SGD optimalizáló rosszul konvergál RNN-eknél
- Nem elég epoch
- Nincs megfelelő learning rate

**Javítás**: Adam optimalizáló, 0.001 tanulási ráta, elegendő epoch.

In [ ]:
def evaluate_model(model, data_loader, device):
    """Modell kiértékelése: pontosság és veszteség számítása."""
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for data, labels in data_loader:
            data, labels = data.to(device), labels.to(device)
            outputs = model(data)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * data.size(0)
            predictions = (torch.sigmoid(outputs) > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total, total_loss / total


def train_model(model, train_loader, val_loader, device, epochs=30, lr=0.001):
    """Modell tanítása Adam optimalizálóval."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)  # Adam SGD helyett!

    train_losses = []
    val_accs = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        for data, labels in train_loader:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            # Gradiens vágás a stabilitásért
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        train_losses.append(avg_loss)

        val_acc, val_loss = evaluate_model(model, val_loader, device)
        val_accs.append(val_acc)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs}: "
                  f"Tanítási veszteség = {avg_loss:.4f}, "
                  f"Validációs pontosság = {val_acc:.4f}")

    return train_losses, val_accs

In [ ]:
# Modell tanítása javított beállításokkal
model_rnn = SimpleRNN(hidden_size=64).to(device)

print("=== Javított tanítás (Adam, lr=0.001) ===")
train_losses, val_accs = train_model(model_rnn, train_loader, val_loader, device, epochs=30, lr=0.001)

# Végső kiértékelés
test_acc, test_loss = evaluate_model(model_rnn, test_loader, device)
print(f"\nTeszt pontosság: {test_acc:.4f}")
print(f"Teszt veszteség: {test_loss:.4f}")

In [ ]:
# Tanulási görbe vizualizáció
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Veszteség')
ax1.set_title('Tanítási veszteség')
ax1.grid(True)

ax2.plot(val_accs)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Pontosság')
ax2.set_title('Validációs pontosság')
ax2.axhline(y=0.9, color='r', linestyle='--', label='90% cél')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

---
## 4. feladat: Hosszabb szekvenciák (5 pont)

Vizsgáljuk meg, hogyan teljesít a 40 hosszú szekvenciákon tanított modell hosszabb szekvenciákon!

In [ ]:
# Különböző szekvenciahosszakon tesztelés
seq_lengths = [40, 60, 80, 120, 160, 240, 320]
accuracies = []

for sl in seq_lengths:
    test_ds = create_dataset(200, 200, sl, NEEDLE_LEN)
    test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)
    acc, _ = evaluate_model(model_rnn, test_dl, device)
    accuracies.append(acc)
    print(f"Szekvenciahossz: {sl:>4d} -> Pontosság: {acc:.4f}")

# Vizualizáció
plt.figure(figsize=(8, 5))
plt.plot(seq_lengths, accuracies, 'bo-', markersize=8, linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', label='Véletlenszerű tippelés')
plt.xlabel('Szekvencia hossza')
plt.ylabel('Pontosság')
plt.title('SimpleRNN pontossága különböző szekvenciahosszakon\n(40-es hosszon tanítva)')
plt.legend()
plt.grid(True)
plt.ylim(0.4, 1.05)
plt.show()

print("\nMegfigyelés: Az RNN teljesítménye jellemzően romlik hosszabb szekvenciáknál,")
print("mert nehezen tud hosszú távú függőségeket megtanulni.")

---
## 5. feladat: Más RNN architektúrák (5 pont)

Hasonlítsuk össze az RNN, LSTM és GRU architektúrákat!

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)  # LSTM h_n és c_n-t is ad vissza
        out = self.fc(h_n[-1])
        return out


class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.gru(x)
        out = self.fc(h_n[-1])
        return out

In [ ]:
# Mindhárom modell tanítása
models = {
    'RNN': SimpleRNN(hidden_size=64).to(device),
    'LSTM': LSTMModel(hidden_size=64).to(device),
    'GRU': GRUModel(hidden_size=64).to(device),
}

results = {}
for name, model in models.items():
    print(f"\n=== {name} tanítása ===")
    train_losses, val_accs = train_model(model, train_loader, val_loader, device, epochs=30, lr=0.001)
    test_acc, _ = evaluate_model(model, test_loader, device)
    results[name] = {'val_accs': val_accs, 'test_acc': test_acc, 'model': model}
    print(f"{name} teszt pontosság: {test_acc:.4f}")

In [ ]:
# Tanulási görbék összehasonlítása
plt.figure(figsize=(8, 5))
for name, res in results.items():
    plt.plot(res['val_accs'], label=f"{name} (teszt: {res['test_acc']:.3f})")
plt.xlabel('Epoch')
plt.ylabel('Validációs pontosság')
plt.title('RNN vs LSTM vs GRU - Validációs pontosság')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Hosszabb szekvenciákon összehasonlítás
seq_lengths = [40, 60, 80, 120, 160, 240, 320]

plt.figure(figsize=(10, 6))
for name, res in results.items():
    accs = []
    for sl in seq_lengths:
        test_ds = create_dataset(200, 200, sl, NEEDLE_LEN)
        test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)
        acc, _ = evaluate_model(res['model'], test_dl, device)
        accs.append(acc)
    plt.plot(seq_lengths, accs, 'o-', markersize=6, linewidth=2, label=name)

plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Véletlenszerű')
plt.xlabel('Szekvencia hossza')
plt.ylabel('Pontosság')
plt.title('Architektúrák összehasonlítása különböző szekvenciahosszakon')
plt.legend()
plt.grid(True)
plt.ylim(0.4, 1.05)
plt.show()

print("Megfigyelés: Az LSTM és GRU jellemzően jobban teljesít hosszabb szekvenciákon,")
print("mert képesek hosszabb távú függőségeket megtanulni a kapuzó mechanizmusoknak köszönhetően.")

---
## 6. feladat: Attribúció / Gradiensek (10 pont)

Vizsgáljuk meg, mely pozíciókra figyel a modell! A gradiens-alapú attribúció megmutatja, hogy az egyes bemeneti pozíciók mennyire befolyásolják a kimenetet.

In [ ]:
def compute_input_gradients(model, x):
    """Bemeneti gradiensek kiszámítása a modell kimenetéhez képest.
    
    Args:
        model: A betanított modell
        x: Bemeneti tenzor (1, seq_len, 1)
    Returns:
        Gradiensek abszolút értéke (seq_len,)
    """
    model.eval()
    x = x.clone().detach().requires_grad_(True)
    output = model(x)
    output.backward()
    # A gradiens abszolút értéke mutatja az egyes pozíciók fontosságát
    gradients = x.grad.squeeze().abs()
    return gradients.detach().cpu().numpy()

In [ ]:
# Pozitív minta generálása ismert tű pozícióval
needle = torch.tensor([int(i % 2 == 0) for i in range(NEEDLE_LEN)], dtype=torch.float32)

# Egy pozitív minta létrehozása, ahol tudjuk a tű pozícióját
seq_len_test = 80
needle_pos = 30  # A tű a 30. pozíciótól kezdődik
test_seq = torch.randint(0, 2, (seq_len_test,), dtype=torch.float32)
test_seq[needle_pos:needle_pos+NEEDLE_LEN] = needle
test_input = test_seq.unsqueeze(0).unsqueeze(-1).to(device)  # (1, seq_len, 1)

# Használjuk az LSTM modellt (általában jobb attribúciót ad)
best_model = results['LSTM']['model']
grads = compute_input_gradients(best_model, test_input)

# Vizualizáció
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 1. A bemeneti sorozat
axes[0].bar(range(seq_len_test), test_seq.numpy(), color='steelblue', alpha=0.7)
axes[0].axvspan(needle_pos, needle_pos + NEEDLE_LEN, alpha=0.2, color='red', label=f'Tű pozíció ({needle_pos}-{needle_pos+NEEDLE_LEN})')
axes[0].set_ylabel('Érték')
axes[0].set_title('Bemeneti sorozat')
axes[0].legend()

# 2. Gradiensek
colors = ['red' if needle_pos <= i < needle_pos + NEEDLE_LEN else 'steelblue' for i in range(seq_len_test)]
axes[1].bar(range(seq_len_test), grads, color=colors, alpha=0.7)
axes[1].axvspan(needle_pos, needle_pos + NEEDLE_LEN, alpha=0.1, color='red')
axes[1].set_ylabel('|Gradiens|')
axes[1].set_title('Bemeneti gradiensek (piros = tű pozíció)')

# 3. Heatmap
axes[2].imshow(grads.reshape(1, -1), aspect='auto', cmap='hot', interpolation='nearest')
axes[2].axvline(x=needle_pos, color='cyan', linewidth=2, linestyle='--')
axes[2].axvline(x=needle_pos + NEEDLE_LEN, color='cyan', linewidth=2, linestyle='--')
axes[2].set_xlabel('Pozíció')
axes[2].set_title('Gradiens heatmap (ciánkék vonalak = tű határai)')
axes[2].set_yticks([])

plt.tight_layout()
plt.show()

# Összehasonlítás: tű vs nem-tű pozíciók átlagos gradiense
needle_grads = grads[needle_pos:needle_pos+NEEDLE_LEN].mean()
non_needle_grads = np.concatenate([grads[:needle_pos], grads[needle_pos+NEEDLE_LEN:]]).mean()
print(f"Átlagos gradiens a tű pozíciókon: {needle_grads:.6f}")
print(f"Átlagos gradiens a többi pozíción: {non_needle_grads:.6f}")
print(f"Arány: {needle_grads/max(non_needle_grads, 1e-10):.2f}x")

In [ ]:
# Több mintán átlagolva
n_samples = 20
avg_grads = np.zeros(seq_len_test)
actual_needle_pos = needle_pos  # Mindig ugyanoda tesszük a tűt

for _ in range(n_samples):
    seq = torch.randint(0, 2, (seq_len_test,), dtype=torch.float32)
    seq[actual_needle_pos:actual_needle_pos+NEEDLE_LEN] = needle
    inp = seq.unsqueeze(0).unsqueeze(-1).to(device)
    g = compute_input_gradients(best_model, inp)
    avg_grads += g

avg_grads /= n_samples

plt.figure(figsize=(12, 4))
colors = ['red' if actual_needle_pos <= i < actual_needle_pos + NEEDLE_LEN else 'steelblue' for i in range(seq_len_test)]
plt.bar(range(seq_len_test), avg_grads, color=colors, alpha=0.7)
plt.axvspan(actual_needle_pos, actual_needle_pos + NEEDLE_LEN, alpha=0.1, color='red', label='Tű pozíció')
plt.xlabel('Pozíció')
plt.ylabel('Átlagos |gradiens|')
plt.title(f'Átlagos gradiens attribúció ({n_samples} mintán)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 7. feladat: Konvolúció érvek (5 pont)

**Miért lenne alkalmas egy CNN a feladatra?**

A tű mintázat (`101010...`) egy **fix hosszúságú, lokális mintázat** a bemeneti sorozatban. A konvolúciós neurális hálózat (CNN) pont ilyen lokális mintázatok felismerésére lett kitalálva. Egy 1D konvolúciós réteg, amelynek kernel mérete legalább akkora, mint a tű hossza, képes egyetlen konvolúciós művelettel felismerni a tű jelenlétét. A CNN-nek nem kell megjegyeznie a sorozat korábbi elemeit (szemben az RNN-nel), hanem egyszerűen végigcsúsztatja a szűrőt a bemeneten, és mindenhol megvizsgálja, hogy a lokális mintázat megegyezik-e a tűvel. Ráadásul a CNN természeténél fogva jól általánosít hosszabb szekvenciákra is, hiszen ugyanazt a szűrőt alkalmazza a sorozat minden pozíciójára, függetlenül a teljes sorozat hosszától. Ez a transzlációs invariancia teszi különösen alkalmassá a feladatra.

---
## 8. feladat: CNN implementáció (5 pont)

1D CNN implementáció a tű keresésére.

In [ ]:
class NeedleCNN(nn.Module):
    """1D konvolúciós háló a tű felismerésére.
    
    A Conv1d kernel mérete legalább a tű hossza, így képes
    egyetlen konvolúciós lépésben felismerni a mintázatot.
    """
    def __init__(self, needle_len=20, n_filters=32):
        super().__init__()
        # Első konvolúciós réteg: kernel méret = tű hossza
        self.conv1 = nn.Conv1d(1, n_filters, kernel_size=needle_len, padding=0)
        self.relu = nn.ReLU()
        # Második konvolúciós réteg kisebb kernellel
        self.conv2 = nn.Conv1d(n_filters, 16, kernel_size=3, padding=1)
        # Global max pooling + lineáris réteg
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        # x: (batch, seq_len, 1) -> (batch, 1, seq_len) a Conv1d-hez
        x = x.transpose(1, 2)
        x = self.relu(self.conv1(x))   # (batch, n_filters, seq_len - needle_len + 1)
        x = self.relu(self.conv2(x))   # (batch, 16, ...)
        # Global max pooling: a szekvencia mentén maximumot vesz
        x = x.max(dim=2)[0]            # (batch, 16)
        x = self.fc(x)                 # (batch, 1)
        return x

In [ ]:
# CNN tanítása
model_cnn = NeedleCNN(needle_len=NEEDLE_LEN, n_filters=32).to(device)

print("=== CNN tanítása ===")
cnn_train_losses, cnn_val_accs = train_model(model_cnn, train_loader, val_loader, device, epochs=30, lr=0.001)

# Teszt pontosság
cnn_test_acc, cnn_test_loss = evaluate_model(model_cnn, test_loader, device)
print(f"\nCNN teszt pontosság: {cnn_test_acc:.4f}")

In [ ]:
# CNN vs RNN vs LSTM vs GRU összehasonlítás különböző szekvenciahosszakon
seq_lengths = [40, 60, 80, 120, 160, 240, 320]

all_models = {
    'RNN': results['RNN']['model'],
    'LSTM': results['LSTM']['model'],
    'GRU': results['GRU']['model'],
    'CNN': model_cnn,
}

plt.figure(figsize=(10, 6))
for name, model in all_models.items():
    accs = []
    for sl in seq_lengths:
        test_ds = create_dataset(200, 200, sl, NEEDLE_LEN)
        test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)
        acc, _ = evaluate_model(model, test_dl, device)
        accs.append(acc)
    plt.plot(seq_lengths, accs, 'o-', markersize=6, linewidth=2, label=name)

plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Véletlenszerű')
plt.xlabel('Szekvencia hossza')
plt.ylabel('Pontosság')
plt.title('Összes architektúra összehasonlítása különböző szekvenciahosszakon')
plt.legend()
plt.grid(True)
plt.ylim(0.4, 1.05)
plt.show()

print("Megfigyelés: A CNN jól általánosít hosszabb szekvenciákra,")
print("mivel a konvolúciós szűrő lokálisan működik és szekvenciahossz-független.")

---
---
# Mistral API rész

A következő feladatokban a Mistral LLM-et teszteljük "Needle in a Haystack" módszerrel.

In [ ]:
# Mistral API inicializálás
MISTRAL_AVAILABLE = False

try:
    from mistralai import Mistral
    api_key = os.environ.get("MISTRAL_API_KEY", "")
    if api_key:
        client = Mistral(api_key=api_key)
        MISTRAL_AVAILABLE = True
        print("Mistral API sikeresen inicializálva.")
    else:
        print("MISTRAL_API_KEY nincs beállítva. Az API feladatok szimulált eredményekkel futnak.")
except Exception as e:
    print(f"Mistral API nem elérhető: {e}")
    print("Az API feladatok szimulált eredményekkel futnak.")

In [ ]:
def query_mistral(prompt, model_name="mistral-small-latest"):
    """Mistral API lekérdezés wrapper."""
    if not MISTRAL_AVAILABLE:
        return None
    try:
        response = client.chat.complete(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256,
            temperature=0.0,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"API hiba: {e}")
        return None

---
## Mistral 1. feladat: NIAH teszt tervezés (5 pont)

### A teszt leírása

A "Needle in a Haystack" (NIAH) tesztben egy specifikus tényt (a "tűt") helyezünk el egy hosszú szövegben (a "szénakazalban"), majd megkérdezzük a nyelvi modellt erről a tényről. A cél annak vizsgálata, hogy a modell képes-e megtalálni és visszaadni a releváns információt a hosszú kontextusból.

**Módszertan:**

1. **A tű**: Egy konkrét, egyedi tény, amely nem valószínű, hogy a modell tanítóadatában szerepel. Például: "A legjobb pizza Budapesten a Tűzrakó Étteremben található, amelynek titkos hozzávalója a szarvasgombás olívaolaj."

2. **A szénakazal**: Angol nyelvű szöveg, amelyet Paul Graham esszéiből vagy hasonló forrásból generálunk. A szöveg hossza változó (pl. 500, 1000, 2000, 4000, 8000 token).

3. **A tű elhelyezése**: A tűt a szöveg különböző pozícióiba illesztjük be (eleje, 25%, közép, 75%, vége), hogy megvizsgáljuk, a pozíció befolyásolja-e a visszakeresés sikerességét.

4. **A kérdés**: A tű tartalmával kapcsolatos kérdést teszünk fel, például: "What is the secret ingredient of the best pizza in Budapest?"

5. **Kiértékelés**: A válasz helyességét vizsgáljuk (tartalmazza-e a kulcsszót, pl. "truffle olive oil" vagy "szarvasgombás olívaolaj"). Az eredményeket heatmap formájában ábrázoljuk: x-tengely a tű pozíciója, y-tengely a kontextus hossza.

---
## Mistral 2. feladat: NIAH implementáció (10 pont)

In [ ]:
# A "tű" - egy specifikus tény
NEEDLE_FACT = "The best pizza in Budapest can be found at the Tuzrako Restaurant, whose secret ingredient is truffle olive oil."
NEEDLE_QUESTION = "What is the secret ingredient of the best pizza in Budapest?"
NEEDLE_ANSWER_KEYWORD = "truffle olive oil"

# Szénakazal szöveg generáló (ismétlődő bekezdések)
FILLER_PARAGRAPHS = [
    "Technology has transformed the way we communicate and interact with the world around us. From smartphones to artificial intelligence, innovation continues to reshape every aspect of our daily lives. The pace of change shows no signs of slowing down.",
    "Education plays a crucial role in shaping the future of society. By investing in quality learning experiences, we can empower individuals to reach their full potential. Critical thinking and problem-solving skills are more important than ever.",
    "The natural world offers endless sources of inspiration and wonder. From the depths of the ocean to the peaks of the highest mountains, our planet is home to an incredible diversity of life forms. Conservation efforts are essential to preserve this biodiversity.",
    "Art and culture have been fundamental to human expression throughout history. Music, literature, painting, and dance allow us to explore emotions and ideas that transcend language barriers. Creative expression enriches our understanding of the human experience.",
    "Scientific research continues to push the boundaries of human knowledge. Breakthroughs in medicine, physics, and biology have the potential to solve some of the world's most pressing challenges. Collaboration between disciplines accelerates discovery.",
    "Urban development presents both opportunities and challenges for modern societies. As cities grow, planners must balance economic growth with environmental sustainability. Smart city initiatives leverage technology to improve quality of life for residents.",
    "The global economy is interconnected in ways that previous generations could not have imagined. Trade agreements, digital commerce, and international cooperation drive economic growth across borders. Understanding these dynamics is essential for informed decision-making.",
    "Health and wellness have become central concerns in contemporary culture. People are increasingly aware of the importance of nutrition, exercise, and mental health. Preventive healthcare approaches aim to address issues before they become serious problems.",
]


def generate_haystack(target_word_count):
    """Szénakazal szöveg generálása megadott szószámmal."""
    paragraphs = []
    word_count = 0
    i = 0
    while word_count < target_word_count:
        p = FILLER_PARAGRAPHS[i % len(FILLER_PARAGRAPHS)]
        paragraphs.append(p)
        word_count += len(p.split())
        i += 1
    return paragraphs


def insert_needle(paragraphs, needle, position_ratio):
    """A tű beillesztése a megadott relatív pozícióba.
    
    Args:
        paragraphs: Bekezdések listája
        needle: A beillesztendő tény
        position_ratio: 0.0 (eleje) és 1.0 (vége) közötti szám
    Returns:
        Összeillesztett szöveg a tűvel
    """
    insert_idx = int(len(paragraphs) * position_ratio)
    insert_idx = max(0, min(insert_idx, len(paragraphs)))
    paragraphs_with_needle = paragraphs[:insert_idx] + [needle] + paragraphs[insert_idx:]
    return "\n\n".join(paragraphs_with_needle)


def check_answer(response, keyword):
    """Ellenőrzi, hogy a válasz tartalmazza-e a kulcsszót."""
    if response is None:
        return False
    return keyword.lower() in response.lower()

In [ ]:
def run_niah_test(model_name="mistral-small-latest", word_counts=None, positions=None):
    """NIAH teszt futtatása különböző szöveghosszakra és tű pozíciókra.
    
    Returns:
        results: 2D numpy tömb (word_counts x positions), 1 = helyes, 0 = helytelen
    """
    if word_counts is None:
        word_counts = [200, 500, 1000, 2000, 4000]
    if positions is None:
        positions = [0.0, 0.25, 0.5, 0.75, 1.0]

    results = np.zeros((len(word_counts), len(positions)))

    for i, wc in enumerate(word_counts):
        for j, pos in enumerate(positions):
            paragraphs = generate_haystack(wc)
            full_text = insert_needle(paragraphs, NEEDLE_FACT, pos)
            
            prompt = f"""Read the following text carefully and answer the question at the end.

---
{full_text}
---

Question: {NEEDLE_QUESTION}
Answer concisely:"""

            if MISTRAL_AVAILABLE:
                response = query_mistral(prompt, model_name=model_name)
                success = check_answer(response, NEEDLE_ANSWER_KEYWORD)
                results[i, j] = 1.0 if success else 0.0
                status = "HELYES" if success else "HELYTELEN"
                print(f"  Szavak: {wc:>5}, Pozíció: {pos:.2f} -> {status}")
            else:
                # Szimulált eredmény: rövid szövegeknél és szélső pozícióknál jobb
                base_prob = max(0.3, 1.0 - wc / 8000)
                pos_penalty = 0.1 * abs(pos - 0.5)  # Közepén nehezebb
                results[i, j] = 1.0 if np.random.random() < (base_prob - pos_penalty) else 0.0

    return results, word_counts, positions


print("=== NIAH teszt futtatása (természetes nyelvű) ===")
if not MISTRAL_AVAILABLE:
    print("(Szimulált eredmények - Mistral API nem elérhető)")

niah_results, word_counts, positions = run_niah_test()
print("\nKész!")

In [ ]:
# NIAH heatmap vizualizáció
def plot_niah_heatmap(results, word_counts, positions, title="NIAH Teszt Eredmények"):
    """NIAH eredmények megjelenítése heatmap formában."""
    fig, ax = plt.subplots(figsize=(8, 6))
    
    im = ax.imshow(results, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1,
                   interpolation='nearest')
    
    # Tengely feliratok
    pos_labels = [f"{int(p*100)}%" for p in positions]
    ax.set_xticks(range(len(positions)))
    ax.set_xticklabels(pos_labels)
    ax.set_yticks(range(len(word_counts)))
    ax.set_yticklabels([str(wc) for wc in word_counts])
    
    ax.set_xlabel('Tű pozíciója a szövegben')
    ax.set_ylabel('Szöveg hossza (szavak)')
    ax.set_title(title)
    
    # Értékek kiírása a cellákba
    for i in range(len(word_counts)):
        for j in range(len(positions)):
            text = "OK" if results[i, j] > 0.5 else "X"
            color = "white" if results[i, j] < 0.5 else "black"
            ax.text(j, i, text, ha="center", va="center", color=color, fontweight='bold')
    
    plt.colorbar(im, label='Sikeres (1) / Sikertelen (0)')
    plt.tight_layout()
    plt.show()


plot_niah_heatmap(niah_results, word_counts, positions,
                  title="NIAH Teszt - Természetes nyelv (Mistral)")

---
## Mistral 3. feladat: Kódolós NIAH tervezés (5 pont)

### A kódolós NIAH teszt leírása

A kódolós NIAH tesztben a "tű" egy specifikus változó-értékadás egy hosszú kódrészletben. A "szénakazal" generált Python kódból áll, amely számos változó-definíciót, függvényt és osztályt tartalmaz. A tű egy egyedi, specifikus értékadás, amelyet a modellnek kell megtalálnia.

**Módszertan:**

1. **A tű**: Egy specifikus változó-értékadás, például `secret_answer = 42.7913`. Ez egy egyedi érték, amely nem szerepel máshol a kódban.

2. **A szénakazal**: Generált Python kód, amely tartalmaz véletlenszerű változó-definíciókat, egyszerű függvényeket, listákat és szótárakat. A kód szintaktikailag helyes, de szemantikailag értelmetlen.

3. **A tű elhelyezése**: A tűt a kód különböző pozícióiba illesztjük be, hasonlóan a természetes nyelvű teszthez.

4. **A kérdés**: "What is the value of the variable `secret_answer`?"

5. **Kiértékelés**: A válasz helyességét a pontos érték alapján ellenőrizzük.

---
## Mistral 4. feladat: Kódolós NIAH implementáció (10 pont)

In [ ]:
import random
import string

# A kódolós tű
CODE_NEEDLE = "secret_answer = 42.7913"
CODE_QUESTION = "What is the value of the variable `secret_answer` in the code above?"
CODE_ANSWER_KEYWORD = "42.7913"


def generate_random_varname():
    """Véletlenszerű változónév generálása."""
    prefixes = ['data', 'result', 'value', 'count', 'total', 'temp', 'var',
                'num', 'idx', 'flag', 'config', 'param', 'score', 'weight']
    return f"{random.choice(prefixes)}_{random.randint(1, 999)}"


def generate_code_line():
    """Véletlenszerű kódsor generálása."""
    templates = [
        lambda: f"{generate_random_varname()} = {random.uniform(-100, 100):.4f}",
        lambda: f"{generate_random_varname()} = {random.randint(-1000, 1000)}",
        lambda: f"{generate_random_varname()} = [{', '.join(str(random.randint(0, 100)) for _ in range(random.randint(3, 7)))}]",
        lambda: f"{generate_random_varname()} = '{random.choice(['alpha', 'beta', 'gamma', 'delta', 'epsilon', 'zeta'])}'",
        lambda: f"# {random.choice(['Compute', 'Calculate', 'Initialize', 'Update', 'Process'])} {generate_random_varname()}",
        lambda: f"{generate_random_varname()} = {generate_random_varname()} + {random.uniform(0, 10):.2f}" if random.random() > 0.5 else f"{generate_random_varname()} = True",
    ]
    return random.choice(templates)()


def generate_code_block(n_lines=10):
    """Kódblokk generálása n sorral."""
    lines = [generate_code_line() for _ in range(n_lines)]
    return "\n".join(lines)


def generate_code_haystack(target_lines):
    """Kód szénakazal generálása megadott sorszámmal."""
    blocks = []
    current_lines = 0
    while current_lines < target_lines:
        block_size = random.randint(5, 15)
        blocks.append(generate_code_block(block_size))
        current_lines += block_size
    return blocks


def insert_code_needle(blocks, needle, position_ratio):
    """A tű beillesztése a kódba a megadott pozícióba."""
    insert_idx = int(len(blocks) * position_ratio)
    insert_idx = max(0, min(insert_idx, len(blocks)))
    blocks_with_needle = blocks[:insert_idx] + [needle] + blocks[insert_idx:]
    return "\n\n".join(blocks_with_needle)

In [ ]:
def run_code_niah_test(model_name="mistral-small-latest", line_counts=None, positions=None):
    """Kódolós NIAH teszt futtatása."""
    if line_counts is None:
        line_counts = [50, 100, 200, 400, 800]
    if positions is None:
        positions = [0.0, 0.25, 0.5, 0.75, 1.0]

    results = np.zeros((len(line_counts), len(positions)))

    for i, lc in enumerate(line_counts):
        for j, pos in enumerate(positions):
            blocks = generate_code_haystack(lc)
            full_code = insert_code_needle(blocks, CODE_NEEDLE, pos)
            
            prompt = f"""Read the following Python code carefully and answer the question.

```python
{full_code}
```

Question: {CODE_QUESTION}
Answer with just the value:"""

            if MISTRAL_AVAILABLE:
                response = query_mistral(prompt, model_name=model_name)
                success = check_answer(response, CODE_ANSWER_KEYWORD)
                results[i, j] = 1.0 if success else 0.0
                status = "HELYES" if success else "HELYTELEN"
                print(f"  Sorok: {lc:>5}, Pozíció: {pos:.2f} -> {status}")
            else:
                # Szimulált eredmény
                base_prob = max(0.2, 1.0 - lc / 1500)
                pos_penalty = 0.15 * abs(pos - 0.0)  # Elején könnyebb
                results[i, j] = 1.0 if np.random.random() < (base_prob - pos_penalty) else 0.0

    return results, line_counts, positions


print("=== Kódolós NIAH teszt futtatása ===")
if not MISTRAL_AVAILABLE:
    print("(Szimulált eredmények - Mistral API nem elérhető)")

code_niah_results, line_counts, positions = run_code_niah_test()
print("\nKész!")

In [ ]:
# Kódolós NIAH heatmap
plot_niah_heatmap(code_niah_results, line_counts, positions,
                  title="Kódolós NIAH Teszt Eredmények (Mistral)")

---
## Mistral 5. feladat: Modell összehasonlítás (5 pont)

Két Mistral modell összehasonlítása a NIAH teszteken: Mistral Small (7B) vs Mistral Large (Mixtral).

In [ ]:
# Modell összehasonlítás
model_configs = {
    "Mistral Small (7B)": "mistral-small-latest",
    "Mistral Large (Mixtral)": "mistral-large-latest",
}

# Kisebb teszt a költségek csökkentése érdekében
comparison_word_counts = [200, 500, 1000, 2000]
comparison_positions = [0.0, 0.25, 0.5, 0.75, 1.0]

comparison_results = {}

for display_name, model_id in model_configs.items():
    print(f"\n=== {display_name} ({model_id}) ===")
    if not MISTRAL_AVAILABLE:
        print("(Szimulált eredmények)")
    
    results, wc, pos = run_niah_test(
        model_name=model_id,
        word_counts=comparison_word_counts,
        positions=comparison_positions
    )
    comparison_results[display_name] = results

In [ ]:
# Összehasonlító heatmapek
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (display_name, results) in enumerate(comparison_results.items()):
    ax = axes[idx]
    im = ax.imshow(results, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1,
                   interpolation='nearest')
    
    pos_labels = [f"{int(p*100)}%" for p in comparison_positions]
    ax.set_xticks(range(len(comparison_positions)))
    ax.set_xticklabels(pos_labels)
    ax.set_yticks(range(len(comparison_word_counts)))
    ax.set_yticklabels([str(wc) for wc in comparison_word_counts])
    
    ax.set_xlabel('Tű pozíciója')
    ax.set_ylabel('Szöveg hossza (szavak)')
    ax.set_title(display_name)
    
    # Értékek a cellákba
    for i in range(len(comparison_word_counts)):
        for j in range(len(comparison_positions)):
            text = "OK" if results[i, j] > 0.5 else "X"
            color = "white" if results[i, j] < 0.5 else "black"
            ax.text(j, i, text, ha="center", va="center", color=color, fontweight='bold')

plt.suptitle('Modell Összehasonlítás - NIAH Teszt', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Összesített pontszámok
print("\n=== Összesített eredmények ===")
for name, results in comparison_results.items():
    avg_score = results.mean()
    print(f"{name}: {avg_score:.1%} sikeres visszakeresés")

---
## Összegzés

### PyTorch rész
- A tanítatlan modell vesztesége ~ln(2), ami véletlenszerű tippelésnek felel meg
- A false negative ráta rendkívül alacsony 20 bites tű esetén
- Adam optimalizáló + megfelelő hiperparaméterek jelentősen javítják az RNN teljesítményét
- Az LSTM és GRU jobban általánosít hosszabb szekvenciákra mint a sima RNN
- A gradiens attribúció képes megmutatni, hol van a tű a sorozatban
- A CNN természetéből adódóan alkalmas lokális mintázatok felismerésére és jól általánosít

### Mistral API rész
- A NIAH teszt hatékonyan méri a kontextus-visszakeresési képességet
- A szöveghossz növekedésével a visszakeresés nehezebb
- A tű pozíciója is befolyásolja az eredményt
- Nagyobb modellek általában jobban teljesítenek a NIAH teszten